In [15]:
## Reading the dataset
data = open('input.txt','r',encoding='utf-8').read()

In [16]:
len(data)

1115394

In [17]:
## first few thousands of characters
print(data[:1000])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.



In [19]:
chars = sorted(list(set(data)))
char_size = len(chars)
print(''.join(chars))
print(char_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [40]:
vocab_size = char_size

In [20]:
## First thing (Think how will you create tokens from the dataset) (as this one is a character level model so we need to represent them in number)
## building encoder (Take string and give out integer) , decoder (Take integer and give out string)

stoi = {s:i for i,s in enumerate(chars)}
itos = {i:s for s, i in stoi.items()}
encoder = lambda s : [stoi[c] for c in s]
decoder = lambda i : ''.join([itos[c] for c in i])

print(encoder('hi, there'))
print(decoder(encoder('hi, there')))

[46, 47, 6, 1, 58, 46, 43, 56, 43]
hi, there


In [35]:
## encoding the entire dataset
import torch
encoData = torch.tensor(encoder(data), dtype=torch.long)
print(encoData.shape , encoData.type)
print(encoData[:1000])

torch.Size([1115394]) <built-in method type of Tensor object at 0x113d305d0>
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59,  1, 39, 56, 43,  1, 39, 50, 50,
         1, 56, 43, 57, 53, 50, 60, 43, 42,  1, 56, 39, 58, 46, 43, 56,  1, 58,
        53,  1, 42, 47, 43,  1, 58, 46, 39, 52,  1, 58, 53,  1, 44, 39, 51, 47,
        57, 46, 12,  0,  0, 13, 50, 50, 10,  0, 30, 43, 57, 53, 50, 60, 43, 42,
         8,  1, 56, 43, 57, 53, 50, 60, 43, 42,  8,  0,  0, 18, 47, 56, 57, 58,
         1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 18, 47, 56, 57, 58,  6,  1, 63,
        53, 59,  1, 49, 52, 53, 61,  1, 15,

In [28]:
## splitting dataset for train and validation
n = int(0.9*len(encoData))

train_data = encoData[:n]
val_data = encoData[n:]

print(train.shape)
print(val.shape)

torch.Size([1003854])
torch.Size([111540])


In [30]:
context_length = 8
encoData[:context_length+1]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [37]:
## Processing data in batch
torch.manual_seed(1337)
batch_size = 4
context_length = 8

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - context_length, (batch_size,))
    x = torch.stack([data[i:i+context_length] for i in ix])
    y = torch.stack([data[i+1:i+context_length+1] for i in ix])
    return x,y

xb , yb = get_batch('train')
print(f'inputs:{xb.shape}')
print(f'outputs:{yb.shape}')

inputs:torch.Size([4, 8])
outputs:torch.Size([4, 8])


In [47]:
## implementing biagram model
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

class BiagramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)
    def forward(self, idx, target=None):
        logits = self.token_embedding_table(idx)
        ## loss
        if target == None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T,C)
            target = target.view(B*T)
            loss = F.cross_entropy(logits, target)
        return logits, loss
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, loss = self(idx)
            logits = logits[:, -1, :] # Focus on last time steps
            # softmwax
            probs = F.softmax(logits, dim=1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

m = BiagramLanguageModel(vocab_size)
out, loss = m.forward(xb, yb)
print(out.shape)
print(loss)

torch.Size([32, 65])
tensor(4.8786, grad_fn=<NllLossBackward0>)


In [48]:
print(decoder(m.generate(torch.zeros((1,1), dtype=torch.long), max_new_tokens=100)[0].tolist()))


SKIcLT;AcELMoTbvZv C?nq-QE33:CJqkOKH-q;:la!oiywkHjgChzbQ?u!3bLIgwevmyFJGUGp
wnYWmnxKWWev-tDqXErVKLgJ


In [49]:
# created a pytorch optimizer
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

In [57]:
batch_size = 32
for steps in range(10000):
    xb, yb = get_batch('train')

    logits, loss = m(xb,yb)
    optimizer.zero_grad(set_to_none = True)
    loss.backward()
    optimizer.step()
print(loss.item())

2.576992988586426


In [59]:
print(decoder(m.generate(torch.zeros((1,1), dtype=torch.long), max_new_tokens=300)[0].tolist()))


IUS: plllyss,
BOricand t me kusotenthayorieanatin inou idis ne horsercoownd
ORGunde ll mugl wrg g be, to fr t athame-he's h f we frs it tepodestha wird ad s; belesuin byes mb. Ond tand m:
Wed toshoths 'd ath! s pouthas COPrgoomur d y; withefeloulorneey mya chile.
SThyof wiorimy heig, ond watomow b,



In [ ]:
## We are seeing some improvment from the last prediction to these new predictions, but not that good, because issue is tokens are not talking to each other
## No relationship between them, the model is only predicting next character based on previous 8 characters, so certainlly it is not good. Let build somthing so that
## Model can talk to each other